In [ ]:
# Purpose - makes fields/spectra movie (newer)

findex = 63 #23 #[26, 63] #[23, 63] #63 #6 #25

In [ ]:
# Imports and Folder Selection

import sys

import numpy as np
import pandas as pd

import xarray as xr

import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from graphet import Data
from graphet.plugins import TristanV2 

sys.path.append('..')
from useful_funcs import *
sys.path.append('../fmgmt')
from TristanRun import TristanRun
#from FieldLineHelper import FieldLineHelper

plt.close('all')
%matplotlib widget
set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)

In [ ]:
# Configure Data

d = run.config_data(run.steps, "fields")

In [ ]:
original_midline = d.fields.isel(x = len(d.fields.x) // 2).isel(z = 0)

In [ ]:
import os

# Specify the location where you want to save the HDF5 file

output_file = f"{TristanRun.find_folderpath(findex)}/output/cut_flds/midline.hdf5"

if os.path.exists(output_file):
    raise FileExistsError(f"The file {output_file} already exists.")

# Write the midline data to the HDF5 file
original_midline.to_netcdf(output_file, engine='h5netcdf')

In [ ]:
# Load the data from the HDF5 file into new_midlines
new_midlines = xr.open_dataset(output_file, engine='h5netcdf')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

for i in range(1):
    run = run
    midline = new_midlines
    ax = [ax1, ax2][i]

    t_max = run.beta_A * run.cc / run.c_omp * run.output_interval * len(midline.t)
    #t_max = len(midline.t)

    im = ax.imshow(np.flipud(midline.dens), aspect='auto', extent=[midline.y.min(), midline.y.max(), 0, t_max], 
                   vmin=0, vmax=10)
    #fig.colorbar(im, ax=ax1, label='$n / n_0$')
    ax.set_xlabel('$y$ [$c / \omega_p$]')
    ax.set_ylabel('$v_A t$ [$c / \omega_p$]')
    ax.set_title(f"{findex} sigma = {run.sigma}")